&nbsp;

# Heat Shock Factor 1 (HSF1) regulation up5p transcripts

In [1]:
import re
import os
import pathlib
import datetime
import subprocess
import polars as pl
from IPython.display import display
from IPython.display import Image
from pygenometracks.readGtf import ReadGtf

pl.Config.set_fmt_str_lengths(200)
os.environ["POLARS_MAX_THREADS"] = "16"

In [2]:
execute_date = datetime.datetime.now().strftime("%y%m%d")  # noqa: DTZ005
print(f"Execute Date: {execute_date}")

Execute Date: 261011


&nbsp;

&nbsp;

&nbsp;

# 1. Basic Information

- [upsetplot を作成した notebook](../06_transcript_characterization/06_6_up5p_lncRNA_intersect.ipynb) で使用したファイルを全て読み込む
- upset plot の結果は、あくまで transcript - cre id ペアに則って分類したものであり、transcript level でみると、実は既に他の要素に入っている場合があるため

In [3]:
up5p_fantabio_all = pl.read_parquet(
    "../../out/fantabio/parquet_file/up5p_transcript_fantabio_v261001.parquet"
)

pl.Config.set_tbl_rows(2)
display(up5p_fantabio_all)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000774581.1""",101,57,248,44,34,44,173,-10,null,null,"""lncRNA""","""ENSG00000300853.1""","[""basic"", ""TAGENE""]","""+""",2,"""URS0002A25289""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_400104""","""chr8""",27592922,"""27593122""","""FCHS_400104""","[""ENST00000774579.1""]","""0""",294,"[{""EH38E3824983"",""dELS""}]",null,"[{""HS_00105821"",""NHEK-Epidermal Keratinocyte Primary Cells"",""Skin"",""Skin"",""SVM""}, {""HS_00105822"",""H1 BMP4 Derived Trophoblast Cultured Cells"",""Trophoblast cell"",""Others"",""SVM""}, … {""HS_00105828"",""Breast variant Human Mammary Epithelial Cells (vHMEC)"",""Breast"",""Breast"",""SVM""}]",null,null,0.5,"""ELA""","""255,202,74""","""yellow"""


In [4]:
# 2. Diff Analysis (ATAC-seq)
up5p_fantabio_diff_analysis = pl.read_parquet(
    "../../out/ChIP-Atlas_diff/diff_analysis_filter_parquet/transcript_cre/fantabio_HN2_up5p_v261002_diff_analysis_atac.parquet"
)

pl.Config.set_tbl_rows(2)
display(up5p_fantabio_diff_analysis)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red""",161523744,161524487,"[0.296097, 0.117093, 0.351279]","[{161524223,161524443,""chr1""}, {161524400,161524568,""chr1""}, {161524226,161524581,""chr1""}]","[1.209197, 1.832055, 1.892709]","[2.7404e-8, 1.8112e-16, 7.5870e-7]","[0.000001, 2.0457e-12, 0.000487]","[""GSE101731_atac_hff_44C_120min"", ""GSE173536_atac_d21_42C_60min"", ""GSE209675_mcf7_42C_60min""]",3,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE173536_atac_d21_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE209675_mcf7_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000678039.1""",102,58,246,44,50,45,156,5,"""RPS20""","""HGNC:10405""","""nonsense_mediated_decay""","""ENSG00000008988.12""","[""RNA_Seq_supported_only"", ""CCDS""]","""-""",2,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported only""]",1,"""FCHS_403770""","""chr8""",56074307,"""56074825""","""cp1@CERNA3,cp2@RPS20""","[""ENST00000676461.1"", ""ENST00000519807.5"", … ""D28358""]","""0""",0,"[{""EH38E2632426"",""pELS""}, {""EH38E3834665"",""PLS""}, {""EH38E3834666"",""PLS""}]","[{""CTCF"",4378,[{""SRX8622036"",4378}, {""SRX10975999"",4357}, … {""SRX102986"",501}]}, {""ZBTB11"",4051,[{""SRX15898392"",4051}, {""SRX2424560"",3514}, … {""SRX2630262"",766}]}, … {""ZNF668"",502,[{""SRX10977254"",502}]}]","[{""HS_00108611"",""HepG2"",""Liver"",""Liver"",""SVM""}, {""HS_00108612"",""Fetal Intestine Large"",""Large intestine"",""Intestine"",""SVM""}, … {""HS_00108615"",""HMEC Mammary Epithelial Primary Cells"",""Mammary gland"",""Breast"",""SVM""}]","[{""hg19::chr8:56987057..56987058,-;hg_93135.1"",""p2@RPS20""}, {""hg19::chr8:56987059..56987060,-;hg_93136.1"",""p3@RPS20""}, … {""hg38::chr8:56074312..56074321,-;hg_231159.1"",""p5@RPS20""}]",null,-0.999715,"""PLA""","""0,0,204""","""blue""",56074306,56074825,[0.119461],"[{56074333,56074395,""chr8"

In [5]:
# 3. Diff Analysis (HSF1)
up5p_fantabio_diff_analysis_hsf1 = pl.read_parquet(
    "../../out/ChIP-Atlas_diff/diff_analysis_filter_parquet/transcript_cre/fantabio_HN2_up5p_v261008_diff_analysis_chipseq_hsf1.parquet"
)

display(up5p_fantabio_diff_analysis_hsf1)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,motif_hit,n_motif_hit,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[struct[9]],u32,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red""",161523744,161524487,"[{""MA0486.2"",""HSF1"",722,734,""+"",14.5323,0.000003,0.41,""TTCGCGAAAGTTC""}, {""MA0486.1"",""HSF1"",696,710,""-"",12.4225,0.0000193,0.383,""CTTCCCGCACCTTCC""}, … {""MA0486.2"",""HSF1"",717,729,""-"",7.74194,0.0000494,0.817,""TTCGCGAACCTTT""}]",8,"[0.363392, 0.54105]","[{161524217,161524537,""chr1""}, {161524085,161524630,""chr1""}]","[3.696223, 3.583503]","[1.2063e-25, 4.3546e-12]","[1.5375e-22, 1.0794e-10]","[""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min"", ""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min""]",2,"[""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000549882.1""",72,28,306,44,36,17,198,19,"""SENP1""","""HGNC:17927""","""retained_intron""","""ENSG00000079387.16""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_93279""","""chr12""",48106429,"""48106892""","""cp14@PFKM""","[""ENST00000548288.5""]","""-41""",363,"[{""EH38E3014683"",""pELS""}, {""EH38E3014684"",""pELS""}, {""EH38E3014685"",""pELS""}]","[{""HSF1"",3992,[{""SRX16646111"",3992}, {""SRX16646095"",3230}, … {""SRX690823"",616}]}, {""MED26"",2539,[{""SRX6384756"",2539}, {""SRX6384755"",2403}]}, … {""EP300"",530,[{""SRX1478537"",530}]}]","[{""HS_00171480"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00171481"",""Gastric"",""Stomach"",""Stomach"",""SVM""}, … {""HS_00171486"",""Pancreas"",""Pancreas"",""Pancreas"",""SVM""}]",null,null,-0.431579,"""ELA""","""255,202,74""","""yellow""",48106428,48106892,"[{""MA0486.1"",""HSF1"",338,352,""-"",20.1831,4.7500e-8,0.0743,""GTTCTGGAAGGTTCT""}, {""MA0486.2"",""HSF1"",339,351,""-"",18.1935,6.7100e-7,0.206,""TTCTGGAAGGTTC""}, {""MA0486.2"",""HSF1"",334,346

In [6]:
up5p_fantabio_cre_divergent = pl.read_parquet(
    "../../out/transcript_classification/divergent_transcript/pc_common_cre_lncRNA_divergent/up5p_fantabio_pc_common_cre_lncRNA_divergent.parquet"
)

display(up5p_fantabio_cre_divergent)

cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,pc_transcript_distance,screen_ccres,fantom5_enhancers,pc_transcript_id,pc_HN2,pc_transcript_strand,pc_transcript_info,lncrna_transcript_distance,lncrna_transcript_id,lncrna_HN2,lncrna_transcript_strand,lncrna_transcript_info,strand_type,pc_lncrna_HN2,id_correspondence_flag,Transcript_classification_codes,FANTOM-CAT_transcript_lv4_stringent,gene class,fantomcat_transcript_info,screen_promoter_support,ensembl_promoter_window_support,fantomcat_stringent_support
str,str,str,str,i64,i64,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5],i64,str,i64,str,struct[5],str,i64,str,str,str,str,list[struct[7]],str,str,str
"""FCHS_110516""","""cp7@HSPH1""","""PLA""","""chr13""",31161745,31162676,-0.981672,0,"[{""EH38E3059049"",""pELS""}, {""EH38E3059050"",""PLS""}, … {""EH38E3059052"",""pELS""}]",null,"""ENST00000954002.1""",263,"""-""","{""ENSG00000120694.21"",""HSPH1"",""protein_coding"",null,[""basic"", ""TAGENE"", ""CCDS""]}",0,"""ENST00000729765.1""",295,"""+""","{""ENSG00000295391.1"",null,""lncRNA"",""URS0002A422B0"",[""basic"", ""TAGENE""]}","""opposite strand""",558,"""4_equal_fantom_native""","""=""","""FTMT25200000861.1""","""uncertain coding""","[{""CATG00000014230.1"",""promoter"",138.6,730.6,null,null,1.3800e-10}]","""supported by SCREEN""","""supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_177259""","""cp1@SPAG9""","""PLA""","""chr17""",51120666,51121355,-0.977787,0,"[{""EH38E3230875"",""PLS""}, {""EH38E3230876"",""pELS""}, {""EH38E4165335"",""PLS""}]","[{[""chr17:51121243-51122181""]}]","""ENST00000933646.1""",44,"""-""","{""ENSG00000008294.23"",""SPAG9"",""protein_coding"",null,[""basic"", ""GENCODE_Primary"", … ""appris_alternative_1""]}",0,"""ENST00000831795.1""",44,"""+""","{""ENSG00000308112.1"",null,""lncRNA"",""URS0002A38AD4"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""opposite strand""",88,"""4_equal_fantom_native""","""=""","""ENCT00000176518.1""","""lncRNA, divergent""","[{""CATG00000031741.1"",""promoter"",347.8,108.8,null,9.7270e-33,5.3900e-42}]","""supported by SCREEN""","""supported by Ensembl/GENCODE promoter window""","""supported by FANTOM-CAT (stringent)"""


In [7]:
e_derived_lncRNA = pl.read_parquet(
    "../../out/transcript_classification/enhancer_derived_lncRNA/up5p_enhancer_derived_lncRNA_v261010.parquet"
)

display(e_derived_lncRNA)

cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean,e_transcript_id,e_HN2,e_transcript_type,e_gene_id,e_gene_symbol,e_strand,e_RNACentral_ID,e_annotation_remarks,e_tags
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64,str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_94451""","""FCHS_94451""","""PLA""","""FCHS_94499""","""cp1@ATG101""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.778816, 0.778816, … 0.904984]",0.767913,0.904984,0.806297,"[0.355003, 0.189876, … 0.526484]",0.189876,0.774307,0.408416,"""ENST00000812391.1""",128,"""lncRNA""","""ENSG00000305684.1""",null,"""-""","""URS0002A236EE""","[""CLS3 project""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_265630""","""cp2@LOC105369306""","""ELA""","""FCHS_265830""","""cp4@RIPPLY3""","""PLA""","[""GSE105028_hESC_42C_60min""]",1,[1.0],1.0,1.0,1.0,[0.27554],0.27554,0.27554,0.27554,"""ENST00000422473.2""",51,"""lncRNA""","""ENSG00000233393.3""",null,"""+""","""URS0002A57053""","[""CLS3 project""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]"


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Upset plot Result

In [8]:
up5p_intersect_result = pl.read_csv(
    "../../out/transcript_classification/upsetplot/up5p_transcript_upset_data_v261010.tsv",
    separator="\t",
    n_threads=16
).with_columns(
    # Split "ENST...-FCHS_..." into transcript_id and cre_id
    pl.col("id")
    .str.split_exact("-", 1)
    .struct.rename_fields(["transcript_id", "cre_id"])
    .alias("id_parts")
).unnest("id_parts")


display(up5p_intersect_result)

all pairs,derived from CLS3 project,Diff Analysis (ATAC-seq),Diff Analysis (HSF1),divergent,enhancer-derived,id,transcript_id,cre_id
bool,bool,bool,bool,bool,bool,str,str,str
true,false,false,false,false,false,"""ENST00000508735.5-FCHS_339438""","""ENST00000508735.5""","""FCHS_339438"""
…,…,…,…,…,…,…,…,…
true,false,false,false,false,true,"""ENST00000660374.1-FCHS_238929""","""ENST00000660374.1""","""FCHS_238929"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. HSF1 regulation + divergent + Diff-Analysis

- 最も注目できる up5p transcripts の組み合わせ

In [9]:
hsf1_divergent_lncrna = up5p_intersect_result.filter(
    (pl.col("Diff Analysis (HSF1)") == True)
    & (pl.col("divergent") == True)
    & (pl.col("Diff Analysis (ATAC-seq)") == True)
    & (pl.col("enhancer-derived") == False)
    # & (pl.col("derived from CLS3 project") == True)
)

display(hsf1_divergent_lncrna)

all pairs,derived from CLS3 project,Diff Analysis (ATAC-seq),Diff Analysis (HSF1),divergent,enhancer-derived,id,transcript_id,cre_id
bool,bool,bool,bool,bool,bool,str,str,str
true,false,true,true,true,false,"""ENST00000665702.1-FCHS_338098""","""ENST00000665702.1""","""FCHS_338098"""
…,…,…,…,…,…,…,…,…
true,true,true,true,true,false,"""ENST00000729764.1-FCHS_110516""","""ENST00000729764.1""","""FCHS_110516"""


In [10]:
hsf1_divergent_lncrna_list = up5p_fantabio_cre_divergent.filter(
    # pair としてフィルタリング
    (pl.col("lncrna_transcript_id").is_in(hsf1_divergent_lncrna.get_column("transcript_id").unique().implode()))
    & (pl.col("cre_id").is_in(hsf1_divergent_lncrna.get_column("cre_id").unique().implode()))
)

pl.Config.set_tbl_rows(20)
print(hsf1_divergent_lncrna_list.select(pl.col("pc_transcript_info").struct.field("gene_symbol").unique()))

pl.Config.set_tbl_rows(2)
display(hsf1_divergent_lncrna_list)

shape: (15, 1)
┌─────────────┐
│ gene_symbol │
│ ---         │
│ str         │
╞═════════════╡
│ IER5        │
│ HSPH1       │
│ DNAJB1      │
│ STIP1       │
│ ZFAND2A     │
│ HSP90AB1    │
│ UBC         │
│ SIRPA       │
│ HSPA4       │
│ TAF7        │
│ FKBP4       │
│ HSPA1B      │
│ ZSCAN31     │
│ TENT5A      │
│ SOD1        │
└─────────────┘


cre_id,cre_name,fantabio_cre_class,cre_chrom,cre_chrom_start,cre_chrom_end,directionality,pc_transcript_distance,screen_ccres,fantom5_enhancers,pc_transcript_id,pc_HN2,pc_transcript_strand,pc_transcript_info,lncrna_transcript_distance,lncrna_transcript_id,lncrna_HN2,lncrna_transcript_strand,lncrna_transcript_info,strand_type,pc_lncrna_HN2,id_correspondence_flag,Transcript_classification_codes,FANTOM-CAT_transcript_lv4_stringent,gene class,fantomcat_transcript_info,screen_promoter_support,ensembl_promoter_window_support,fantomcat_stringent_support
str,str,str,str,i64,i64,f64,i64,list[struct[2]],list[struct[1]],str,i64,str,struct[5],i64,str,i64,str,struct[5],str,i64,str,str,str,str,list[struct[7]],str,str,str
"""FCHS_110516""","""cp7@HSPH1""","""PLA""","""chr13""",31161745,31162676,-0.981672,0,"[{""EH38E3059049"",""pELS""}, {""EH38E3059050"",""PLS""}, … {""EH38E3059052"",""pELS""}]",null,"""ENST00000954002.1""",263,"""-""","{""ENSG00000120694.21"",""HSPH1"",""protein_coding"",null,[""basic"", ""TAGENE"", ""CCDS""]}",0,"""ENST00000729765.1""",295,"""+""","{""ENSG00000295391.1"",null,""lncRNA"",""URS0002A422B0"",[""basic"", ""TAGENE""]}","""opposite strand""",558,"""4_equal_fantom_native""","""=""","""FTMT25200000861.1""","""uncertain coding""","[{""CATG00000014230.1"",""promoter"",138.6,730.6,null,null,1.3800e-10}]","""supported by SCREEN""","""supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_338098""","""cp4@HSPA4""","""PLA""","""chr5""",133051532,133052212,0.972476,0,"[{""EH38E3661015"",""PLS""}]",null,"""ENST00000870719.1""",44,"""+""","{""ENSG00000170606.17"",""HSPA4"",""protein_coding"",null,[""basic"", ""TAGENE""]}",0,"""ENST00000665702.1""",62,"""-""","{""ENSG00000286408.1"",null,""lncRNA"",""URS0000EF3281"",[""basic"", ""Ensembl_canonical"", … ""TAGENE""]}","""opposite strand""",106,null,null,null,null,null,"""supported by SCREEN""","""supported by Ensembl/GENCODE promoter window""","""not supported by FANTOM-CAT (stringent)"""


In [11]:
# metascape でエンリッチメント解析を行うための protein-coding transcript のリストを取得
# metascape execution: 2026-10-11
hsf1_divergent_pc_transcript_list = hsf1_divergent_lncrna_list.select(
    pl.col("pc_transcript_id").unique()
)

# hsf1_divergent_pc_transcript_list.write_csv(
#     "../../out/metascape/transcript_list/up5p_diffanalysis_atac_hsf1/hsf1_divergent_pc_transcript_list.tsv",
#     separator="\t",
#     include_header=False
# )

# display(hsf1_divergent_pc_transcript_list)

In [35]:
# For TogoMCP survey
hsf1_divergent_pc_lncRNA_cre_list = hsf1_divergent_lncrna_list.select(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "pc_transcript_id",
    "pc_transcript_distance",
    "pc_HN2",
    "lncrna_transcript_id",
    "lncrna_transcript_distance",
    "lncrna_HN2",
    "strand_type",
    "pc_lncrna_HN2"
).unique().sort(
    by=["pc_lncrna_HN2"],
    descending=True
)

hsf1_divergent_pc_lncRNA_cre_list.write_csv(
    "../../out/transcript_survey/TogoMCP_survey/hsf1_divergent_pc_lncRNA_common_cre_list.tsv",
    separator="\t",
    include_header=True
)

display(hsf1_divergent_pc_lncRNA_cre_list)

cre_id,cre_name,fantabio_cre_class,pc_transcript_id,pc_transcript_distance,pc_HN2,lncrna_transcript_id,lncrna_transcript_distance,lncrna_HN2,strand_type,pc_lncrna_HN2
str,str,str,str,i64,i64,str,i64,i64,str,i64
"""FCHS_110516""","""cp7@HSPH1""","""PLA""","""ENST00000954002.1""",0,263,"""ENST00000729765.1""",0,295,"""opposite strand""",558
…,…,…,…,…,…,…,…,…,…,…
"""FCHS_338098""","""cp4@HSPA4""","""PLA""","""ENST00000870719.1""",0,44,"""ENST00000665702.1""",0,62,"""opposite strand""",106


In [12]:
hsf1_divergent_lncrna_transcript_list = up5p_fantabio_diff_analysis.filter(
    # pair としてフィルタリング
    (pl.col("transcript_id").is_in(hsf1_divergent_lncrna.get_column("transcript_id").unique().implode()))
    & (pl.col("cre_id").is_in(hsf1_divergent_lncrna.get_column("cre_id").unique().implode()))
)

hsf1_divergent_lncrna_transcript_list.write_parquet(
    "../../out/transcript_survey/divergent_transcript/up5p_hsf1_divergent_lncrna_transcript_list.parquet"
)

display(hsf1_divergent_lncrna_transcript_list)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000729765.1""",315,20,71,295,206,4,41,202,null,null,"""lncRNA""","""ENSG00000295391.1""","[""basic"", ""TAGENE""]","""+""",2,"""URS0002A422B0""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_110516""","""chr13""",31161745,"""31162676""","""cp7@HSPH1""","[""ENST00000954002.1"", ""ENST00000320027.10"", … ""AK311464""]","""0""",0,"[{""EH38E3059049"",""pELS""}, {""EH38E3059050"",""PLS""}, … {""EH38E3059052"",""pELS""}]","[{""RFX1"",4409,[{""SRX2423398"",4409}, {""SRX2424673"",4103}, … {""SRX5457514"",596}]}, {""HSF1"",3960,[{""SRX16646095"",3960}, {""SRX16646102"",3724}, … {""SRX13845688"",503}]}, … {""ZHX2"",507,[{""SRX3636887"",507}]}]","[{""HS_00184065"",""H1 Derived Neuronal Progenitor Cultured Cells"",""Brain"",""Brain"",""SVM""}, {""HS_00184066"",""H1 BMP4 Derived Trophoblast Cultured Cells"",""Trophoblast cell"",""Others"",""SVM""}, … {""HS_00184072"",""Fetal Intestine Small"",""Small intestine"",""Intestine"",""SVM""}]","[{""hg19::chr13:31735993..31736009,-;hg_136690.1"",""p3@HSPH1""}, {""hg19::chr13:31736027..31736053,-;hg_136691.1"",""p1@HSPH1""}, … {""hg19::chr13:31736780..31736803,+;hg_134654.1"",""hg_134654.1""}]",null,-0.981672,"""PLA""","""0,0,204""","""blue""",31161744,31162676,"[0.05794, 0.040773]","[{31161798,31161852,""chr13""}, {31161701,31161782,""chr13""}]","[1.492452, 1.617098]","[5.5610e-10, 2.6045e-10]","[0.000003, 6.8502e-7]","[""GSE173536_atac_d21_42C_60min"", ""GSE173536_atac_t21_42C_60min""]",2,"[""GSE173536_atac_d21_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE173536_atac_t21_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000810160.1""",52,8,346,44,35,5,211,30,null,null,"""lncRNA""","""ENSG00000305295.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""+""",4,"""URS0002A20E69""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_361740""","""chr6""",81753189,"""81753630""","""FCHS_361740""","[""ENST00000810355.1""]","""0""",-381,"[{""EH38E3718362"",""pELS""}]","[{""HSF1"",3676,[{""SRX16646111"",3676}, {""SRX16646095"",2473}, … {""SRX16646094"",821}]}, {""STAT2"",3069,[{""SRX19004919"",3069}, {""SRX19004920"",2875}, … {""SRX22524737"",710}]}, … {""ZBTB8A"",500,[{""SRX2423997"",500}]}]","[{""HS_00078721"",""Gastric"",""Stomach"",""Stomach"",""SVM""}, {""HS_00078722"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, … {""HS_00078725"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]","[{""hg19::chr6:82462885..82462911,-;hg_74535.1"",""p4@TENT5A""}, {""hg19::chr6:82462935..82462946,-;hg_74536.1"",""p3@TENT5A""}]","[{[""chr6:81753106-81754120""]}]",0.364462,"""PLA""","""50,0,0""","""dark_red""",81753188,81753630,[0.223982],"[{81753112,81753287,""chr6""}]",[1.257651],[0.000006],[0.000105],"[""GSE101731_atac_hff_44C_120m

In [ ]:
# survey


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. HSF1-regulated lncRNA + diff-analysis (ATAC-seq) + enhancer-derived 

In [13]:
hsf1_lncrna_enhancer_derived = up5p_intersect_result.filter(
    (pl.col("Diff Analysis (HSF1)") == True)
    & (pl.col("divergent") == False)
    & (pl.col("Diff Analysis (ATAC-seq)") == True)
    & (pl.col("enhancer-derived") == True)
)

display(hsf1_lncrna_enhancer_derived)

all pairs,derived from CLS3 project,Diff Analysis (ATAC-seq),Diff Analysis (HSF1),divergent,enhancer-derived,id,transcript_id,cre_id
bool,bool,bool,bool,bool,bool,str,str,str
true,true,true,true,false,true,"""ENST00000792518.1-FCHS_345096""","""ENST00000792518.1""","""FCHS_345096"""
…,…,…,…,…,…,…,…,…
true,true,true,true,false,true,"""ENST00000842153.1-FCHS_267410""","""ENST00000842153.1""","""FCHS_267410"""


In [14]:
hsf1_lncrna_enhancer_derived_transcript_list = up5p_fantabio_all.filter(
    (pl.col("transcript_id").is_in(hsf1_lncrna_enhancer_derived.get_column("transcript_id").unique().implode()))
    & (pl.col("cre_id").is_in(hsf1_lncrna_enhancer_derived.get_column("cre_id").unique().implode()))
)

display(hsf1_lncrna_enhancer_derived_transcript_list)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000686199.2""",210,24,172,186,118,10,123,108,null,null,"""lncRNA""","""ENSG00000289637.2""","[""basic"", ""TAGENE""]","""-""",2,"""URS0002A2F8A7""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project"", ""Assembled from PacBio CLS3 reads""]",2,"""FCHS_354474""","""chr6""",31821758,"""31822349""","""cp10@SNHG32""","[""ENST00000718216.1"", ""ENST00000718219.1"", … ""ENST00000718221.1""]","""0""",0,"[{""EH38E3700971"",""PLS""}, {""EH38E3700972"",""pELS""}]","[{""NFE2"",2671,[{""SRX2424107"",2671}, {""SRX2424108"",2272}, … {""SRX646135"",749}]}, {""HSF1"",2314,[{""SRX16646111"",2314}, {""SRX6858250"",1772}, … {""SRX262145"",577}]}, … {""STAT1"",507,[{""SRX212650"",507}]}]","[{""HS_00075267"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}, {""HS_00075268"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}]","[{""hg19::chr6:31789738..31789757,-;hg_72842.1"",""hg_72842.1""}, {""hg19::chr6:31789892..31789941,+;hg_67055.1"",""hg_67055.1""}, … {""hg19::chr6:31790120..31790128,+;hg_67059.1"",""hg_67059.1""}]","[{[""chr6:31821895-31822503""]}]",0.554413,"""PLA""","""101,0,0""","""dark_red"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000752769.1""",123,74,209,49,84,29,138,55,null,null,"""lncRNA""","""ENSG00000298060.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""+""",4,"""URS0002A4428A""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_218758""","""chr2""",28389659,"""28390651""","""FCHS_218758""","[""ENST00000752769.1"", ""ENST00000752770.1""]","""0""",0,"[{""EH38E1983507"",""dELS""}, {""EH38E1983508"",""dELS""}, … {""EH38E3333572"",""dELS""}]","[{""ATF4"",2979,[{""SRX9741170"",2979}, {""SRX10900664"",2179}, … {""SRX8510436"",588}]}, {""SPI1"",2077,[{""SRX9317678"",2077}]}, … {""LEF1"",514,[{""SRX10774917"",514}]}]","[{""HS_00023172"",""Foreskin Melanocyte Primary Cells skin01"",""Prepuce"",""Skin"",""SVM""}, {""HS_00023173"",""H1 Derived Neuronal Progenitor Cultured Cells"",""Brain"",""Brain"",""SVM""}, … {""HS_00023178"",""Fetal Brain Female"",""Brain"",""Brain"",""SVM""}]",null,"[{[""chr2:28390180-28391556""]}]",0.649832,"""PLA""","""153,0,0""","""dark_red"""


In [15]:
# check
FCHS_94451 = hsf1_lncrna_enhancer_derived_transcript_list.filter(
    pl.col("cre_id") == "FCHS_94451"
)

display(FCHS_94451)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000812391.1""",144,16,246,128,71,7,173,64,null,null,"""lncRNA""","""ENSG00000305684.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""-""",4,"""URS0002A236EE""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_94451""","""chr12""",52032765,"""52033406""","""FCHS_94451""","[""ENST00000812391.1""]","""0""",0,"[{""EH38E1612154"",""dELS""}]","[{""FLI1"",3270,[{""SRX117010"",3270}, {""SRX117011"",3229}, … {""SRX10144613"",606}]}, {""ASCL1"",3020,[{""SRX8949797"",3020}, {""SRX11245139"",2649}, … {""SRX17737100"",654}]}, … {""RXRA"",501,[{""SRX5457234"",501}]}]","[{""HS_00172322"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, {""HS_00172323"",""Breast variant Human Mammary Epithelial Cells (vHMEC)"",""Breast"",""Breast"",""SVM""}, … {""HS_00172329"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}]","[{""hg19::chr12:52426680..52426686,-;hg_130979.1"",""hg_130979.1""}, {""hg19::chr12:52426731..52426752,-;hg_130980.1"",""hg_130980.1""}, … {""hg19::chr12:52427143..52427164,+;hg_125551.1"",""hg_125551.1""}]","[{[""chr12:52032715-52033380""]}]",-0.114194,"""PLA""","""0,0,0""","""black"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2-1. Main Focus: CRE

- ENCODE-rE2G の予測で、どのような 遺伝子の promoter を標的にしているかをここで調べる

In [16]:
hsf1_lncrna_enhancer_derived_encode_re2g_list = e_derived_lncRNA.filter(
    (pl.col("e_transcript_id").is_in(hsf1_lncrna_enhancer_derived_transcript_list.get_column("transcript_id").unique().implode()))
    & (pl.col("cre_id").is_in(hsf1_lncrna_enhancer_derived_transcript_list.get_column("cre_id").unique().implode()))
)

display(hsf1_lncrna_enhancer_derived_encode_re2g_list)

cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean,e_transcript_id,e_HN2,e_transcript_type,e_gene_id,e_gene_symbol,e_strand,e_RNACentral_ID,e_annotation_remarks,e_tags
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64,str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_94451""","""FCHS_94451""","""PLA""","""FCHS_94499""","""cp1@ATG101""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.778816, 0.778816, … 0.904984]",0.767913,0.904984,0.806297,"[0.355003, 0.189876, … 0.526484]",0.189876,0.774307,0.408416,"""ENST00000812391.1""",128,"""lncRNA""","""ENSG00000305684.1""",null,"""-""","""URS0002A236EE""","[""CLS3 project""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_15797""","""FCHS_15797""","""ELA""","""FCHS_15803""","""cp3@ALG6""","""PLA""","[""GSE105028_hESC_42C_60min""]",1,[1.0],1.0,1.0,1.0,[0.551385],0.551385,0.551385,0.551385,"""ENST00000771247.1""",104,"""lncRNA""","""ENSG00000300378.1""",null,"""-""","""URS0002A2D986""","[""CLS3 project""]","[""TAGENE""]"


In [17]:
# (UP5P) target promoter side transcript candidate
fantabio_HN2_up5p_target_transcript_candidate = up5p_fantabio_all.select(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "transcript_id",
    "HN2",
    "transcript_type",
    "gene_id",
    "gene_symbol",
    "strand",
    "RNACentral_ID",
    "annotation_remarks",
    "tags"
).unique().rename(
    # assign frag name "t" to target promoter side
    {
        "cre_id": "target_cre_id",
        "cre_name": "target_cre_name",
        "fantabio_cre_class": "target_fantabio_cre_class",
        "transcript_id": "t_transcript_id",
        "HN2": "t_HN2",
        "transcript_type": "t_transcript_type",
        "gene_id": "t_gene_id",
        "gene_symbol": "t_gene_symbol",
        "strand": "t_strand",
        "RNACentral_ID": "t_RNACentral_ID",
        "annotation_remarks": "t_annotation_remarks",
        "tags": "t_tags"
    }
)

display(fantabio_HN2_up5p_target_transcript_candidate)

target_cre_id,target_cre_name,target_fantabio_cre_class,t_transcript_id,t_HN2,t_transcript_type,t_gene_id,t_gene_symbol,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags
str,str,str,str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_158620""","""cp1@SLC7A6OS""","""ELA""","""ENST00000562050.6""",100,"""nonsense_mediated_decay""","""ENSG00000132600.19""","""PRMT7""","""+""",null,"[""RNA-Seq supported only""]","[""RNA_Seq_supported_only""]"
…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_130084""","""cp3@SNHG10""","""ELA""","""ENST00000663212.1""",259,"""retained_intron""","""ENSG00000247092.10""","""SNHG10""","""-""",null,null,[]


In [18]:
# (DOWN5P) target promoter side transcript candidate

down5p_fantabio_all = pl.read_parquet(
    "../../out/fantabio/parquet_file/bottom5p_transcript_fantabio_v261001.parquet"
)

fantabio_HN2_down5p_target_transcript_candidate = down5p_fantabio_all.select(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "transcript_id",
    "HN2",
    "transcript_type",
    "gene_id",
    "gene_symbol",
    "strand",
    "RNACentral_ID",
    "annotation_remarks",
    "tags"
).unique().rename(
    # assign frag name "t" to target promoter side
    {
        "cre_id": "target_cre_id",
        "cre_name": "target_cre_name",
        "fantabio_cre_class": "target_fantabio_cre_class",
        "transcript_id": "t_transcript_id",
        "HN2": "t_HN2",
        "transcript_type": "t_transcript_type",
        "gene_id": "t_gene_id",
        "gene_symbol": "t_gene_symbol",
        "strand": "t_strand",
        "RNACentral_ID": "t_RNACentral_ID",
        "annotation_remarks": "t_annotation_remarks",
        "tags": "t_tags"
    }
)

display(fantabio_HN2_down5p_target_transcript_candidate)

target_cre_id,target_cre_name,target_fantabio_cre_class,t_transcript_id,t_HN2,t_transcript_type,t_gene_id,t_gene_symbol,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags
str,str,str,str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_172462""","""cp22@ERBB2""","""ELA""","""ENST00000580074.1""",-68,"""protein_coding""","""ENSG00000141736.15""","""ERBB2""","""+""",null,null,"[""mRNA_start_NF"", ""mRNA_end_NF"", … ""cds_end_NF""]"
…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_415504""","""cp2@MIR6850,cp7@RPL8""","""PLA""","""ENST00000525232.5""",-163,"""retained_intron""","""ENSG00000161016.19""","""RPL8""","""-""",null,null,[]


&nbsp;

&nbsp;

&nbsp;

## 2-1-1: (enhancer) up5p - (target promoter) up5p (called UP-UP)

- 標的となる transcript も今回の解析では up5p transcript に分類されている pattern

In [19]:
hsf1_lncrna_enhancer_derived_encode_re2g_list_target = (
    hsf1_lncrna_enhancer_derived_encode_re2g_list
    .join(
        fantabio_HN2_up5p_target_transcript_candidate,
        on=["target_cre_id", "target_cre_name", "target_fantabio_cre_class"],
        how="inner"
    )
).sort(
    by=["re2g_dataset_count"],
    descending=True
).with_columns(
    # Join enhancer CRE, enhancer transcript, target CRE, and target gene
    pl.concat_str(
        [
            pl.col("cre_id"),
            pl.col("e_transcript_id"),
            # pl.col("target_cre_id"),
            # pl.col("t_gene_id"),
        ],
        separator="-",
    ).alias("cre-e_transcript")
).with_columns(
    # Join enhancer CRE, enhancer transcript, target CRE, and target gene
    pl.concat_str(
        [
            pl.col("cre_id"),
            pl.col("e_transcript_id"),
            pl.col("target_cre_id"),
            # pl.col("t_gene_id"),
        ],
        separator="-",
    ).alias("cre-e_transcript-target_cre")
).with_columns(
    # Join enhancer CRE, enhancer transcript, target CRE, and target gene
    pl.concat_str(
        [
            pl.col("cre_id"),
            pl.col("e_transcript_id"),
            pl.col("target_cre_id"),
            pl.col("t_gene_id"),
        ],
        separator="-",
    ).alias("cre-e_transcript-target_cre-t_gene_id")
)

hsf1_lncrna_enhancer_derived_encode_re2g_list_target.write_parquet(
    "../../out/transcript_survey/enhancer_derived_transcript/up5p_hsf1_enhancer_derived_encode_re2g_list_up5p_up5p.parquet"
)

display(hsf1_lncrna_enhancer_derived_encode_re2g_list_target)

cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean,e_transcript_id,e_HN2,e_transcript_type,e_gene_id,e_gene_symbol,e_strand,e_RNACentral_ID,e_annotation_remarks,e_tags,t_transcript_id,t_HN2,t_transcript_type,t_gene_id,t_gene_symbol,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags,cre-e_transcript,cre-e_transcript-target_cre,cre-e_transcript-target_cre-t_gene_id
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64,str,i64,str,str,str,str,str,list[str],list[str],str,i64,str,str,str,str,str,list[str],list[str],str,str,str
"""FCHS_73669""","""cp2@LINC02736""","""PLA""","""FCHS_73694""","""cp7@MALAT1""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.611247, 0.611247, … 0.811736]",0.611247,1.0,0.696385,"[0.709414, 0.546621, … 0.736767]",0.374908,0.781765,0.594557,"""ENST00000689737.3""",142,"""lncRNA""","""ENSG00000286756.4""",null,"""+""","""URS0002A63BCB""","[""Assembled from PacBio CLS3 reads"", ""CLS3 project""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""ENST00000783184.1""",90,"""lncRNA""","""ENSG00000301979.1""",null,"""-""","""URS0002A58B12""","[""CLS3 project""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""FCHS_73669-ENST00000689737.3""","""FCHS_73669-ENST00000689737.3-FCHS_73694""","""FCHS_73669-ENST00000689737.3-FCHS_73694-ENSG00000301979.1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_85348""","""cp2@LINC02731""","""PLA""","""FCHS_85384""","""cp1@NCAPD3,cp1@VPS26B""","""PLA""","[""GSE267931_HUVEC_39C_120min""]",1,"[1.0, 1.0]",1.0,1.0,1.0,"[0.502913, 0.201]",0.201,0.502913,0.351956,"""ENST00000526377.1""",125,"""lncRNA""","""ENSG00000254648.1""",null,"""-""","""URS000036AAB3""",null,"[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""ENST00000525432.2""",160,"""retained_intron""","""ENSG00000151503.15""","""NCAPD3""","""-""",null,"[""RNA-Seq supported only""]","[""RNA_Seq_supported_only""]","""FCHS_85348-ENST00000526377.1""","""FCHS_85348-ENST00000526377.1-FCHS_85384""","""FCHS_85348-ENST00000526377.1-FCHS_85384-ENSG00000151503.15"""


In [20]:
# check
ENEMAL = hsf1_lncrna_enhancer_derived_encode_re2g_list_target.filter(
    pl.col("e_gene_symbol") == "ENEMAL"
)

display(ENEMAL)

cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean,e_transcript_id,e_HN2,e_transcript_type,e_gene_id,e_gene_symbol,e_strand,e_RNACentral_ID,e_annotation_remarks,e_tags,t_transcript_id,t_HN2,t_transcript_type,t_gene_id,t_gene_symbol,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags,cre-e_transcript,cre-e_transcript-target_cre,cre-e_transcript-target_cre-t_gene_id
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64,str,i64,str,str,str,str,str,list[str],list[str],str,i64,str,str,str,str,str,list[str],list[str],str,str,str


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2-1-2: (enhancer) up5p - (target promoter) down5p (called UP-DOWN)

- 標的となる transcript が、今回の解析では、down5p に含まれるペア
- 念のため、既に UP-UP に含まれていないことも後で確認できるように、flag を付けておく
- 興味深いことに、UP-DOWN のペアの半分くらいは、同じ CRE の 遺伝子レベル で UP-UP と同様の CRE を標的にしている (違うのは transcript だけ)

In [21]:
hsf1_lncrna_enhancer_derived_encode_re2g_list_target_down5p = (
    hsf1_lncrna_enhancer_derived_encode_re2g_list
    .join(
        fantabio_HN2_down5p_target_transcript_candidate,
        on=["target_cre_id", "target_cre_name", "target_fantabio_cre_class"],
        how="inner"
    )
).sort(
    by=["re2g_dataset_count"],
    descending=True
).with_columns(
    # Join enhancer CRE, enhancer transcript, target CRE, and target gene
    pl.concat_str(
        [
            pl.col("cre_id"),
            pl.col("e_transcript_id"),
            # pl.col("target_cre_id"),
            # pl.col("t_gene_id"),
        ],
        separator="-",
    ).alias("cre-e_transcript")
).with_columns(
    # Join enhancer CRE, enhancer transcript, target CRE, and target gene
    pl.concat_str(
        [
            pl.col("cre_id"),
            pl.col("e_transcript_id"),
            pl.col("target_cre_id"),
            # pl.col("t_gene_id"),
        ],
        separator="-",
    ).alias("cre-e_transcript-target_cre")
).with_columns(
    # Join enhancer CRE, enhancer transcript, target CRE, and target gene
    pl.concat_str(
        [
            pl.col("cre_id"),
            pl.col("e_transcript_id"),
            pl.col("target_cre_id"),
            pl.col("t_gene_id"),
        ],
        separator="-",
    ).alias("cre-e_transcript-target_cre-t_gene_id")
)


hsf1_lncrna_enhancer_derived_encode_re2g_list_target_down5p.write_parquet(
    "../../out/transcript_survey/enhancer_derived_transcript/up5p_hsf1_enhancer_derived_encode_re2g_list_up5p_down5p.parquet"
)

display(hsf1_lncrna_enhancer_derived_encode_re2g_list_target_down5p)

cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean,e_transcript_id,e_HN2,e_transcript_type,e_gene_id,e_gene_symbol,e_strand,e_RNACentral_ID,e_annotation_remarks,e_tags,t_transcript_id,t_HN2,t_transcript_type,t_gene_id,t_gene_symbol,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags,cre-e_transcript,cre-e_transcript-target_cre,cre-e_transcript-target_cre-t_gene_id
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64,str,i64,str,str,str,str,str,list[str],list[str],str,i64,str,str,str,str,str,list[str],list[str],str,str,str
"""FCHS_279918""","""FCHS_279918""","""PLA""","""FCHS_279932""","""cp1@HDAC11,cp1@HDAC11-AS1""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",13,"[1.0, 1.0, … 1.0]",0.752244,1.0,0.965336,"[0.64231, 0.222779, … 0.737152]",0.222779,0.87494,0.512844,"""ENST00000773816.1""",64,"""lncRNA""","""ENSG00000300747.1""",null,"""-""","""URS0002A457D3""","[""CLS3 project""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""ENST00000424112.3""",-72,"""lncRNA""","""ENSG00000244502.3""","""HDAC11-AS1""","""-""","""URS0002A30FDE""","[""CLS3 project"", ""GENCODE experimental pipeline""]","[""basic"", ""Ensembl_canonical"", … ""exp_conf""]","""FCHS_279918-ENST00000773816.1""","""FCHS_279918-ENST00000773816.1-FCHS_279932""","""FCHS_279918-ENST00000773816.1-FCHS_279932-ENSG00000244502.3"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_73669""","""cp2@LINC02736""","""PLA""","""FCHS_73510""","""cp12@SLC25A45""","""PLA""","[""GSE173536_lymphblastoid_d21_42C_60min""]",1,[0.801956],0.801956,0.801956,0.801956,[0.27446],0.27446,0.27446,0.27446,"""ENST00000689737.3""",142,"""lncRNA""","""ENSG00000286756.4""",null,"""+""","""URS0002A63BCB""","[""Assembled from PacBio CLS3 reads"", ""CLS3 project""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""ENST00000527174.5""",-61,"""protein_coding""","""ENSG00000162241.14""","""SLC25A45""","""-""",null,null,"[""basic"", ""appris_principal_1"", ""CCDS""]","""FCHS_73669-ENST00000689737.3""","""FCHS_73669-ENST00000689737.3-FCHS_73510""","""FCHS_73669-ENST00000689737.3-FCHS_73510-ENSG00000162241.14"""


In [22]:
# check
hsf1_lncrna_enhancer_derived_encode_re2g_list_target_down5p_filter = hsf1_lncrna_enhancer_derived_encode_re2g_list_target_down5p.filter(
    (
        pl.col("cre-e_transcript-target_cre-t_gene_id").is_in(
            hsf1_lncrna_enhancer_derived_encode_re2g_list_target.get_column("cre-e_transcript-target_cre-t_gene_id").unique().implode()
        )
    )
)

display(hsf1_lncrna_enhancer_derived_encode_re2g_list_target_down5p_filter)

cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean,e_transcript_id,e_HN2,e_transcript_type,e_gene_id,e_gene_symbol,e_strand,e_RNACentral_ID,e_annotation_remarks,e_tags,t_transcript_id,t_HN2,t_transcript_type,t_gene_id,t_gene_symbol,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags,cre-e_transcript,cre-e_transcript-target_cre,cre-e_transcript-target_cre-t_gene_id
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64,str,i64,str,str,str,str,str,list[str],list[str],str,i64,str,str,str,str,str,list[str],list[str],str,str,str
"""FCHS_107551""","""FCHS_107551""","""PLA""","""FCHS_107477""","""cp8@SCARB1""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",13,"[0.883598, 1.0, … 1.0]",0.883598,1.0,0.975376,"[0.841868, 0.857204, … 0.488237]",0.219103,0.857204,0.508386,"""ENST00000776681.1""",92,"""lncRNA""","""ENSG00000301157.1""",null,"""+""","""URS0002A5FA8E""","[""CLS3 project""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""ENST00000877827.1""",-57,"""protein_coding""","""ENSG00000073060.18""","""SCARB1""","""-""",null,"[""DanaFarber""]","[""basic"", ""GENCODE_Primary"", ""TAGENE""]","""FCHS_107551-ENST00000776681.1""","""FCHS_107551-ENST00000776681.1-FCHS_107477""","""FCHS_107551-ENST00000776681.1-FCHS_107477-ENSG00000073060.18"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_85348""","""cp2@LINC02731""","""PLA""","""FCHS_85384""","""cp1@NCAPD3,cp1@VPS26B""","""PLA""","[""GSE267931_HUVEC_39C_120min""]",1,"[1.0, 1.0]",1.0,1.0,1.0,"[0.502913, 0.201]",0.201,0.502913,0.351956,"""ENST00000526377.1""",125,"""lncRNA""","""ENSG00000254648.1""",null,"""-""","""URS000036AAB3""",null,"[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""ENST00000687155.1""",-48,"""retained_intron""","""ENSG00000151503.15""","""NCAPD3""","""-""",null,"[""RNA-Seq supported only""]","[""RNA_Seq_supported_only""]","""FCHS_85348-ENST00000526377.1""","""FCHS_85348-ENST00000526377.1-FCHS_85384""","""FCHS_85348-ENST00000526377.1-FCHS_85384-ENSG00000151503.15"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 3. Only HSF1-regulated lncRNA

- 他の要素をもたない lncRNA

In [23]:
only_hsf1_lncrna = up5p_intersect_result.filter(
    (pl.col("Diff Analysis (HSF1)") == True)
    & (pl.col("divergent") == False)
    & (pl.col("Diff Analysis (ATAC-seq)") == False)
    & (pl.col("enhancer-derived") == False)
)

display(only_hsf1_lncrna)

all pairs,derived from CLS3 project,Diff Analysis (ATAC-seq),Diff Analysis (HSF1),divergent,enhancer-derived,id,transcript_id,cre_id
bool,bool,bool,bool,bool,bool,str,str,str
true,true,false,true,false,false,"""ENST00000824599.1-FCHS_61530""","""ENST00000824599.1""","""FCHS_61530"""
…,…,…,…,…,…,…,…,…
true,true,false,true,false,false,"""ENST00000776419.1-FCHS_442331""","""ENST00000776419.1""","""FCHS_442331"""


In [31]:
only_hsf1_lncrna_list = up5p_fantabio_diff_analysis_hsf1.filter(
    # pair としてフィルタリング
    (pl.col("transcript_id").is_in(only_hsf1_lncrna.get_column("transcript_id").unique().implode()))
    & (pl.col("cre_id").is_in(only_hsf1_lncrna.get_column("cre_id").unique().implode()))
).filter(
    pl.col("RNACentral_ID").is_not_null()
).join(
    up5p_fantabio_cre_divergent.select(pl.col("lncrna_transcript_id").unique()),
    left_on="transcript_id",
    right_on="lncrna_transcript_id",
    how="anti",
).join(
    e_derived_lncRNA.select(pl.col("e_transcript_id").unique()),
    left_on="transcript_id",
    right_on="e_transcript_id",
    how="anti",
)

only_hsf1_lncrna_list.write_parquet(
    "../../out/transcript_survey/only_hsf1/only_hsf1_lncrna_list.parquet"
)


pl.Config.set_tbl_rows(2)
display(only_hsf1_lncrna_list)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,motif_hit,n_motif_hit,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[struct[9]],u32,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000654233.1""",223,25,158,198,163,18,70,145,"""OLMALINC""","""HGNC:28060""","""lncRNA""","""ENSG00000235823.6""","[""basic"", ""TAGENE""]","""+""",2,"""URS0000EF02B3""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","[""Assembled from PacBio CLS reads - CAGE_xor_PolyA_HiSeq (GSE93848)"", ""Assembled from SLRseq reads (SRP049776)""]",2,"""FCHS_57568""","""chr10""",100373144,"""100373826""","""cp6@OLMALINC""","[""ENST00000768998.1"", ""ENST00000769002.1"", … ""BC024924""]","""0""",0,"[{""EH38E1493490"",""PLS""}, {""EH38E2920135"",""pELS""}, {""EH38E4018751"",""PLS""}]","[{""NFYB"",3294,[{""SRX8536118"",3294}, {""SRX037419"",2863}, … {""SRX16608146"",506}]}, {""SREBF1"",3090,[{""SRX150706"",3090}, {""SRX2423588"",2863}, … {""SRX9531418"",607}]}, … {""EGR2"",502,[{""SRX2424611"",502}]}]","[{""HS_00143417"",""H1 Derived Neuronal Progenitor Cultured Cells"",""Brain"",""Brain"",""SVM""}, {""HS_00143418"",""Pancreas"",""Pancreas"",""Pancreas"",""SVM""}, … {""HS_00143421"",""Gastric"",""Stomach"",""Stomach"",""SVM""}]","[{""hg19::chr10:102133061..102133107,-;hg_111056.1"",""hg_111056.1""}, {""hg19::chr10:102133271..102133305,+;hg_106580.1"",""p4@OLMALINC""}, … {""hg19::chr10:102133567..102133578,+;hg_106585.1"",""p7@OLMALINC""}]",null,0.828865,"""PLA""","""204,0,0""","""red""",100373143,100373826,"[{""MA0486.1"",""HSF1"",560,574,""-"",11.7746,0.0000286,0.426,""CGTCTGGAAAACTCT""}]",1,[0.349927],"[{100373587,100374206,""chr10""}]",[1.982803],[0.00014],[0.000862],"[""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min""]",1,"[""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000608396.2""",71,27,308,44,46,9,196,37,"""OLMALINC""","""HGNC:28060""","""lncRNA""","""ENSG00000235823.6""","[""RNA_Seq_supported_only""]","""+""",1,"""URS0000BC63E9""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","[""RNA-Seq supported only""]",1,"""FCHS_57570""","""chr10""",100373869,"""100374348""","""cp2@OLMALINC""","[""ENST00000664516.1""]","""0""",-325,"[{""EH38E1493493"",""pELS""}, {""EH38E2920137"",""pELS""}]","[{""ATF4"",3455,[{""SRX1041183"",3455}, {""SRX9741170"",3226}, … {""SRX7785390"",514}]}, {""HSF1"",3399,[{""SRX16646111"",3399}, {""SRX16646095"",1780}, … {""SRX16646102"",580}]}, … {""SP1"",514,[{""SRX5457222"",514}]}]","[{""HS_00143417"",""H1 Derived Neuronal Progenitor Cultured Cells"",""Brain"",""Brain"",""SVM""}, {""HS_00143418"",""Pancreas"",""Pancreas"",""Pancreas"",""SVM""}, … {""HS_00143421"",""Gastric"",""Stomach"",""Stomach"",""SVM""}]","[{""hg19::chr10:102133591..102133639,+;hg_106586.1"",""p1@OLMALINC""}, {""hg19::chr10:102133736..102133786,-;hg_111057.1"",""hg_111057.1""}]",null,0.313077,"""PLA""","""50,0,0""","""dark_red""",100373868,100374348,"[{""MA0486.2"",""HSF1"",174,186,""+"",11.8548,0.000011,0.628,""TTCAGGAACGTTC""}, {""MA0486.1"

In [30]:
# survey
only_hsf1_lncrna_list_filter = only_hsf1_lncrna_list.filter(
    pl.col("HGNC_ID").is_null()
)

pl.Config.set_tbl_rows(4)
display(only_hsf1_lncrna_list_filter)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,motif_hit,n_motif_hit,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[struct[9]],u32,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000839207.1""",203,50,153,153,139,19,93,120,null,null,"""lncRNA""","""ENSG00000309168.1""","[""TAGENE""]","""+""",1,"""URS0002A6C9FE""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_154112""","""chr16""",29469747,"""29469908""","""FCHS_154112""","[""ENST00000839214.1"", ""ENST00000839213.1"", … ""ENST00000839207.1""]","""0""",0,"[{""EH38E3176074"",""pELS""}]","[{""HSF1"",4367,[{""SRX16646095"",4367}, {""SRX16646094"",3791}, … {""SRX155755"",605}]}, {""RBPJ"",4050,[{""SRX4374941"",4050}, {""SRX4374902"",3920}, … {""SRX4374903"",2357}]}, … {""AHR"",507,[{""SRX20282969"",507}]}]",null,null,null,0.89,"""ELA""","""255,202,74""","""yellow""",29469746,29469908,"[{""MA0486.1"",""HSF1"",135,149,""-"",12.2676,0.0000212,0.394,""TTTCTGGAATTTTGC""}, {""MA0486.2"",""HSF1"",136,148,""-"",9.29032,0.0000308,0.727,""TTCTGGAATTTTG""}, … {""MA0486.1"",""HSF1"",140,154,""+"",10.0141,0.000073,0.516,""ATTCCAGAAAGCTCA""}]",4,"[0.759259, 1.0, 0.580247]","[{29469739,29469869,""chr16""}, {29469709,29470115,""chr16""}, {29469814,29470001,""chr16""}]","[2.940085, 3.377133, 1.258202]","[0.000002, 6.6261e-11, 0.000739]","[0.000017, 1.2752e-9, 0.006806]","[""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min"", ""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min"", ""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min""]",3,"[""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000693438.3""",141,21,244,120,91,5,155,86,null,null,"""lncRNA""","""ENSG00000289301.3""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""-""",4,"""URS0002A5EB92""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""Assembled from PacBio CLS3 reads"", ""CLS3 project""]",2,"""FCHS_57568""","""chr10""",100373144,"""100373826""","""cp6@OLMALINC""","[""ENST00000768998.1"", ""ENST00000769002.1"", … ""BC024924""]","""0""",0,"[{""EH38E1493490"",""PLS""}, {""EH38E2920135"",""pELS""}, {""EH38E4018751"",""PLS""}]","[{""NFYB"",3294,[{""SRX8536118"",3294}, {""SRX037419"",2863}, … {""SRX16608146"",506}]}, {""SREBF1"",3090,[{""SRX150706"",3090}, {""SRX2423588"",2863}, … {""SRX9531418"",607}]}, … {""EGR2"",502,[{""SRX2424611"",502}]}]","[{""HS_00143417"",""H1 Derived Neuronal Progenitor Cultured Cells"",""Brain"",""Brain"",""SVM""}, {""HS_00143418"",""Pancreas"",""Pancreas"",""Pancreas"",""SVM""}, … {""HS_00143421"",""Gastric"",""Stomach"",""Stomach"",""SVM""}]","[{""hg19::chr10:102133061..102133107,-;hg_111056.1"",""hg_111056.1""}, {""hg19::chr10:102133271..102133305,+;hg_106580.1"",""p4@OLMALINC""}, … {""hg19::chr10:102133567..102133578,+;hg_106585.1"",""p7@OLMALINC""}]",null,0.828865,"""PLA""","""204,0,0""","""red""",100373143,100373826,"[{""MA0486.1"",""HSF1"",560,574,""-"",11.7